### Install Requirements
##### Use python version 3.10.17

In [ ]:
!uv pip install -r requirements.txt

### Data Cleaning
#### Fill Missing Values

In [1]:
import pandas as pd
from typing import Union, Optional, Sequence, Tuple

def filter_by_ids(
    static_df: pd.DataFrame,
    ts_df: pd.DataFrame,
    static_id_col: str,
    ts_id_col: str,
    n_ids: int,
    max_ts_rows: Optional[int] = None,
    random_state: Optional[int] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Selects n_ids from static_df[static_id_col], filters both DataFrames,
    and (optionally) limits ts_df to max_ts_rows PER ID.

    Parameters
    ----------
    static_df : pd.DataFrame
        The 'static conditions' table containing unique IDs.
    ts_df : pd.DataFrame
        The 'time series' table keyed by the same IDs.
    static_id_col : str
        Column name in static_df holding the ID.
    ts_id_col : str
        Column name in ts_df holding the same ID values.
    n_ids : int
        Number of unique IDs to sample.
    max_ts_rows : int, optional
        Maximum number of rows **per ID** to return from the filtered ts_df,
        based on its existing order.
    random_state : int, optional
        Seed for reproducibility when sampling.

    Returns
    -------
    filtered_static : pd.DataFrame
        Rows from static_df matching the sampled IDs.
    filtered_ts : pd.DataFrame
        Filtered rows from ts_df, then head(max_ts_rows) per group if requested.
    """
    # 1. Sample unique IDs
    unique_ids = static_df[static_id_col].drop_duplicates()
    if n_ids > len(unique_ids):
        raise ValueError(
            f"n_ids ({n_ids}) exceeds available unique IDs ({len(unique_ids)})"
        )
    sampled_ids = unique_ids.sample(n=n_ids, random_state=random_state).tolist()

    # 2. Filter both DataFrames by sampled IDs
    filtered_static = static_df[static_df[static_id_col].isin(sampled_ids)].reset_index(drop=True)
    filtered_ts     = ts_df[ts_df[ts_id_col].isin(sampled_ids)]

    # 3. If requested, cap rows per ID group
    if max_ts_rows is not None:
        # head(n) on a GroupBy returns first n rows of each group
        # preserving original order :contentReference[oaicite:0]{index=0}
        filtered_ts = (
            filtered_ts
            .groupby(ts_id_col, group_keys=False)
            .head(max_ts_rows)
            .reset_index(drop=True)
        )  # :contentReference[oaicite:1]{index=1}

    return filtered_static, filtered_ts

def fill_missing_values_df(
    df: pd.DataFrame,
    cat_fill: str = "categorical_missing",
    num_fill: Union[int, float] = 0,
    date_fill: str = "01/01/2100",
    date_cols: Optional[Sequence[str]] = None
) -> pd.DataFrame:
    """
    Fill missing values in an existing DataFrame:
      - numeric columns → `num_fill`
      - categorical columns → `cat_fill`
      - specified date columns → `date_fill`
    Any remaining NaNs are dropped. Optionally, save to CSV.

    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame to process.
    cat_fill : str
        Replacement value for missing entries in non-numeric, non-date columns.
    num_fill : int or float
        Replacement value for missing entries in numeric columns.
    date_fill : str
        Date string (D/M/YYYY) to use for filling missing dates.
    date_cols : list of str, optional
        Columns to treat as dates (parsed with dayfirst=True).
    output_csv : str or Path, optional
        If provided, writes the resulting DataFrame to this path.

    Returns
    -------
    pd.DataFrame
        The processed DataFrame with missing values filled and any rows
        with remaining NaNs dropped.
    """
    # Work on a copy to avoid mutating the original
    df = df.copy()

    # 1. Handle date columns first
    if date_cols:
        default_ts = pd.to_datetime(date_fill, dayfirst=True)  # :contentReference[oaicite:0]{index=0}
        for col in date_cols:
            ts = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
            df[col] = ts.fillna(default_ts).dt.strftime("%-d/%-m/%Y")  # :contentReference[oaicite:1]{index=1}

    # 2. Identify numeric vs. categorical (excluding date_cols)
    df_num = df.apply(pd.to_numeric, errors="ignore")
    num_cols = df_num.select_dtypes(include="number").columns
    cat_cols = [
        c for c in df.columns
        if c not in num_cols and (not date_cols or c not in date_cols)
    ]

    # 3. Fill missing values
    df[num_cols] = df[num_cols].fillna(num_fill)          # :contentReference[oaicite:2]{index=2}
    df[cat_cols] = df[cat_cols].fillna(cat_fill)          # :contentReference[oaicite:3]{index=3}

    # 4. Drop any remaining missing values
    df = df.dropna()

    return df

In [2]:
# Read CSV
static_path = "/scratch/synthetic-ts-long/dataset/kevin_baseball/players.csv" # Fill your static dataset path
ts_path = "/scratch/synthetic-ts-long/dataset/kevin_baseball/batting.csv" # Fill your ts dataset path

static_df = pd.read_csv(static_path)
ts_df = pd.read_csv(ts_path)

In [3]:
static_df, ts_df = filter_by_ids(
    static_df=static_df,
    ts_df=ts_df,
    static_id_col='players_id',
    ts_id_col='players_id',
    n_ids=100,
    max_ts_rows=1000,
    random_state=42
)

In [4]:
# Fill in missing values
## Choose what to fill in for categorical, numerical, datetime cols, provide list of date_cols and set output csv path
new_static_df = fill_missing_values_df(
    df = static_df,
    cat_fill = "categorical_missing",
    num_fill = 0,
    date_fill = "01/01/2100",
    date_cols = None
)

new_ts_df = fill_missing_values_df(
    df = ts_df,
    cat_fill = "categorical_missing",
    num_fill = 0,
    date_fill = "01/01/2100",
    date_cols = None
)

/tmp/ipykernel_3754617/9743127.py:113: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_num = df.apply(pd.to_numeric, errors="ignore")
/tmp/ipykernel_3754617/9743127.py:113: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_num = df.apply(pd.to_numeric, errors="ignore")


In [5]:
new_static_df.to_csv(static_path.replace(".csv", "_no_missing.csv"), index=False)
new_ts_df.to_csv(ts_path.replace(".csv", "_no_missing.csv"), index=False)

### Data Preprocessing
#### For non conditional synthesis: Set static_data_path and static_id to None
#### For conditional synthesis: Set static_data_path to static conditions csv path and static_id to column name used as ID

### If you want to freeze conditions during conditional generation then set freeze conditions to True. To freeze the conditions only use rtf-r in config

In [6]:
import os

static_data_path = "/scratch/synthetic-ts-long/dataset/kevin_baseball/players_no_missing.csv"   #The static condition data csv path, if existing else set to None
time_series_csv = "/scratch/synthetic-ts-long/dataset/kevin_baseball/batting_no_missing.csv"    #The original time series data csv path.

static_id = "players_id"        #The static condition id column name in the time series dataset if providing static conditions else set to None
output_dir = "/scratch/synthetic-ts-long/dataset/kevin_baseball/result"         #The output directory for the trained model.
data_dir = os.path.join(os.path.dirname(time_series_csv), "time_series")

In [7]:
import pandas as pd
df = pd.read_csv(time_series_csv)

os.makedirs(data_dir, exist_ok=True)

if static_id not in df.columns:
    raise ValueError(f"Column '{static_id}' not found in the CSV file")

unique_series = df[static_id].unique()

# Split and save
series_files = {}
file_lengths = {}
base_name = os.path.splitext(os.path.basename(time_series_csv))[0]
for series in unique_series:
    series_file = os.path.join(data_dir, f"{series}.csv")
    
    data = df[df[static_id] == series]
    if len(data) > 1:
        data.to_csv(series_file, index=False)
        series_files[series] = series_file
        file_lengths[series] = len(data)

static_df = pd.read_csv(static_data_path) if static_data_path else None
new_series = list(series_files.keys())
if static_df is not None:
    static_df = static_df[static_df[static_id].isin(new_series)]
    static_base_dir = os.path.dirname(static_data_path)
    static_file_name = os.path.splitext(os.path.basename(static_data_path))[0] + "_processed.csv"
    static_data_path = os.path.join(static_base_dir, static_file_name)
    static_df.to_csv(static_data_path, index=False)

In [8]:
print(f"Split completed: {len(series_files)} files created")
print(f"Maximum file lengths: {max(file_lengths.values())}")
print(f"Minimum file lengths: {min(file_lengths.values())}")

Split completed: 67 files created
Maximum file lengths: 20
Minimum file lengths: 2


### Train Model

In [18]:
# Set configurations based on your dataset size, examples in the config folder
config = "./config/rtfr-small.yaml"              #The config file path for the model.

In [19]:
if static_data_path == None:
    command = f"python main.py -t -o {output_dir} -d {data_dir} -c {config}"
else:
    command = f"python main.py -t -o {output_dir} -d {data_dir} -s {static_data_path} -i {static_id} -c {config}"
print(f"Running command: {command}")
!{command}

Running command: python main.py -t -o /scratch/synthetic-ts-long/dataset/kevin_baseball/result -d /scratch/synthetic-ts-long/dataset/kevin_baseball/time_series -s /scratch/synthetic-ts-long/dataset/kevin_baseball/players_no_missing_processed.csv -i players_id -c ./config/rtfr-small.yaml


Fitting time series columns: 100%|██████████████| 14/14 [00:01<00:00, 13.61it/s]
fracts.dataset.transformer [INFO][2025-06-27 08:47:51,093]: Fitting transformer time: 0:00:01
Aggregating: 100%|██████████████████████████████| 14/14 [00:00<00:00, 15.13it/s]
fracts.dataset.transformer [INFO][2025-06-27 08:47:52,086]: Getting static (normalized=False) time: 0:00:00
Computing the sensitivity threshold...
Using parallel computation!!!
Bootstrap round: 100%|████████████████████████| 500/500 [00:05<00:00, 96.75it/s]
Sensitivity threshold summary:
count    500.000000
mean       0.341333
std        0.088532
min        0.058333
25%        0.278842
50%        0.342347
75%        0.403571
max        0.571154
dtype: float64
Sensitivity threshold: 0.4848239295718287 qt_max: 0.05
Map: 100%|█████████████████████████████| 67/67 [00:00<00:00, 5619.81 examples/s]
{'train_runtime': 0.614, 'train_samples_per_second': 109.125, 'train_steps_per_second': 1.629, 'train_loss': 5.650886058807373, 'epoch': 1.0}
10

### Generate Data

In [20]:
freeze_conditions = False

In [21]:
num_samples = 5
if static_data_path == None:
    command = f"python main.py -g -o {output_dir} -d {data_dir} -c {config} -n {num_samples} -a"
else:
    if not freeze_conditions:
        command = f"python main.py -g -o {output_dir} -d {data_dir} -s {static_data_path} -i {static_id} -c {config} -n {num_samples} -a"
    else:
        command = f"python main.py -g -o {output_dir} -d {data_dir} -s {static_data_path} -i {static_id} -c {config} -n {num_samples}"
!{command}

Loading parent model from: /scratch/synthetic-ts-long/dataset/kevin_baseball/result/static/final-static/id000017510140792087296000
128it [00:00, 276.78it/s]                                                       
Generated 0 invalid samples out of total 128 samples generated. Sampling efficiency is: 100.0000%
  country.val birthDate.val  ... throws.val number_of_child
0         USA    1994-08-12  ...          R               1
1      Mexico    1990-08-25  ...          R               1
2      Mexico    1990-08-25  ...          R               1
3      Mexico    1943-03-03  ...          L               1
4      Mexico    1987-06-13  ...          R               1

[5 rows x 10 columns]
Loading child model from: /scratch/synthetic-ts-long/dataset/kevin_baseball/result/static/final-agg/id000017510140813073725440
Config of the encoder: <class 'transformers.models.gpt2.modeling_gpt2.GPT2Model'> is overwritten by shared encoder config: GPT2Config {
  "activation_function": "gelu_new",
  "arch